[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ReevesJustin/data-driven-reloading/blob/main/notebooks/10_When_Is_A_Result_Real.ipynb)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ReevesJustin/data-driven-reloading/main?filepath=notebooks/10_When_Is_A_Result_Real.ipynb)

Time to complete: 10-15 minutes

# When IS a Result Real? — Power Analysis and Confidence Intervals

**Goal:** Learn to distinguish real effects from noise using statistical tools.


In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed
import scipy.stats as stats
from statsmodels.stats.power import TTestIndPower

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


ModuleNotFoundError: No module named 'seaborn'

## 1. The Question You Should Always Ask

In reloading, we often see small differences between loads: "Primer A gives 5 fps more than Primer B." But is this difference *real*, or could it be due to random chance?

Statistical hypothesis testing helps us answer this. We set up a null hypothesis (H0: no difference) and alternative (H1: there is a difference). We collect data and see if the results are unlikely under H0.

The key question: **Before declaring a winner, ask: 'Could this difference have occurred by chance alone?'**

<div style="border: 2px solid black; padding: 10px; margin: 10px 0; background-color: #f9f9f9;">
**Bold Takeaway:** Always consider the possibility of random variation. Small samples can fool you into thinking there's a real effect when there isn't.
</div>

## 2. What is Statistical Power?

Statistical power is the probability of correctly rejecting the null hypothesis when it's false (detecting a real effect).

There are two types of errors:
- **Type I Error (False Positive):** Rejecting H0 when it's true (α, significance level).
- **Type II Error (False Negative):** Failing to reject H0 when it's false (β, 1 - power).

Power = 1 - β. Higher power means less chance of missing a real effect.

Factors affecting power: sample size (n), effect size (δ), significance level (α).


In [ ]:
# Interactive power simulation
def plot_power(n=30, effect_size=0.5, alpha=0.05):
    # Null distribution
    x = np.linspace(-3, 3, 1000)
    null_dist = stats.norm.pdf(x, 0, 1)
    
    # Alternative distribution
    alt_dist = stats.norm.pdf(x, effect_size, 1)
    
    # Critical value for rejection
    crit_val = stats.norm.ppf(1 - alpha)
    
    # Power: area under alternative to the right of crit_val
    power = 1 - stats.norm.cdf(crit_val, effect_size, 1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(x, null_dist, label='Null Hypothesis (H0)', color='blue')
    plt.plot(x, alt_dist, label='Alternative Hypothesis (H1)', color='red')
    plt.axvline(crit_val, color='black', linestyle='--', label=f'Critical Value (α={alpha})')
    plt.fill_between(x, 0, alt_dist, where=(x > crit_val), color='red', alpha=0.3, label=f'Power = {power:.2f}')
    plt.fill_between(x, 0, null_dist, where=(x > crit_val), color='blue', alpha=0.3, label=f'α = {alpha}')
    plt.xlabel('Test Statistic')
    plt.ylabel('Density')
    plt.title(f'Power Analysis: n={n}, Effect Size={effect_size}, α={alpha}')
    plt.legend()
    plt.show()

# Widget
interact(plot_power, n=(10, 100, 10), effect_size=(0.1, 2.0, 0.1), alpha=(0.01, 0.10, 0.01))


<div style="border: 2px solid black; padding: 10px; margin: 10px 0; background-color: #f9f9f9;">
**Bold Takeaway:** Increase sample size or effect size to boost power. Low power means you might miss real differences.
</div>

## 3. Confidence Intervals: Your Uncertainty Made Visible

A confidence interval (CI) gives a range where the true parameter likely lies, with a certain probability (e.g., 95%).

For means, CI width depends on sample size: larger n → narrower CI.

Small samples give wide intervals — be skeptical of 'precise' results from few shots.


In [ ]:
# Simulate CI for different sample sizes
def plot_ci(sample_size=30):
    np.random.seed(42)
    data = np.random.normal(100, 10, sample_size)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = stats.t.interval(0.95, sample_size-1, mean, se)
    
    plt.figure(figsize=(10, 6))
    plt.errorbar(1, mean, yerr=(ci[1]-ci[0])/2, fmt='o', capsize=5, label=f'95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]')
    plt.axhline(100, color='red', linestyle='--', label='True Mean')
    plt.xlim(0.5, 1.5)
    plt.xticks([1], [f'n={sample_size}'])
    plt.ylabel('Value')
    plt.title(f'Confidence Interval for Sample Size {sample_size}')
    plt.legend()
    plt.show()

# Widget
interact(plot_ci, sample_size=(5, 100, 5))


<div style="border: 2px solid black; padding: 10px; margin: 10px 0; background-color: #f9f9f9;">
**Bold Takeaway:** Small samples give wide intervals — be skeptical of 'precise' results from few shots.
</div>

## 4. The Magic Number Calculator

Calculate the sample size needed to achieve desired power.

Formula (approximate for two-sample t-test): n = ((z_α + z_β) / δ)^2 * 2

Where δ is effect size (difference / SD).


In [ ]:
# Sample size calculator
def calc_sample_size(confidence=0.95, effect_size=0.5):
    alpha = 1 - confidence
    power_analysis = TTestIndPower()
    sample_size = power_analysis.solve_power(effect_size=effect_size, alpha=alpha, power=0.8, alternative='two-sided')
    print(f"Required sample size per group: {int(np.ceil(sample_size))}")
    print(f"Formula approximation: n ≈ (({stats.norm.ppf(1-alpha/2)} + {stats.norm.ppf(0.8)}) / {effect_size})^2")
    approx = ((stats.norm.ppf(1-alpha/2) + stats.norm.ppf(0.8)) / effect_size)**2
    print(f"Approximate n: {int(np.ceil(approx))}")

# Widget
interact(calc_sample_size, confidence=(0.8, 0.99, 0.01), effect_size=(0.1, 2.0, 0.1))


## 5. Real Example: Is Primer A Really Better Than Primer B?

Simulate: Primer A: μ=2850, σ=10; Primer B: μ=2860, σ=10. Sample size n=20 each.

Perform t-test, show p-value, CI, power.


In [ ]:
# Simulate data
np.random.seed(42)
n = 20
primer_a = np.random.normal(2850, 10, n)
primer_b = np.random.normal(2860, 10, n)

# t-test
t_stat, p_val = stats.ttest_ind(primer_a, primer_b)
print(f"t-statistic: {t_stat:.3f}, p-value: {p_val:.3f}")

# Confidence interval for difference
diff = np.mean(primer_b) - np.mean(primer_a)
se_diff = np.sqrt(np.var(primer_a)/n + np.var(primer_b)/n)
ci_diff = stats.t.interval(0.95, 2*n-2, diff, se_diff)
print(f"Mean difference: {diff:.2f} fps")
print(f"95% CI: [{ci_diff[0]:.2f}, {ci_diff[1]:.2f}]")

# Power analysis
effect_size = (2860 - 2850) / 10  # 1 SD
power_analysis = TTestIndPower()
power = power_analysis.power(effect_size=effect_size, nobs1=n, alpha=0.05, alternative='two-sided')
print(f"Power: {power:.2f}")

# Plot
plt.figure(figsize=(10, 6))
sns.boxplot(data=[primer_a, primer_b], notch=True)
plt.xticks([0, 1], ['Primer A', 'Primer B'])
plt.ylabel('Velocity (fps)')
plt.title('Velocity Comparison')
plt.show()


Discussion: With n=20, power is moderate. If p<0.05, the difference might be real, but check CI and power.

<div style="border: 2px solid black; padding: 10px; margin: 10px 0; background-color: #f9f9f9;">
**Bold Takeaway:** Always check power and CIs. A 'significant' p-value doesn't mean the effect is practically important.
</div>

> **Key Takeaways**
> - Statistical significance testing determines if results are real
> - P-values and confidence levels quantify result reliability
> - Type I and Type II errors affect conclusion validity
> - Power analysis ensures adequate test sensitivity
> - Proper statistical methods validate or refute claims

[Previous: 10_When_IS_a_Result_Real.ipynb](10_When_IS_a_Result_Real.ipynb) | [Next: 11_Peer_Review_Your_Own_Data.ipynb](11_Peer_Review_Your_Own_Data.ipynb)